# ARC2 matched task-local adaptation pilot

Fifty fixed public training tasks; same GPU/checkpoint, tokenization, grammar and identity/transpose predictions before and after eight demonstration-only LoRA steps. Reset adapters between tasks. Public training labels are read only after both prediction files are frozen. Not a competition submission or unseen evaluation. Maximum 12 hours including startup and a ten-minute finalization reserve.

In [ ]:
from pathlib import Path
import json,time,sys,subprocess
START=time.monotonic()
END=START+43200-600
sources={'arc2/__init__.py': '"""Separate ARC-AGI-2 experiments; no dependency on the ARC-AGI-3 runtime."""\n', 'arc2/symbolic.py': '"""Bounded task-local program search; uses only the task\'s demonstration pairs."""\nfrom collections import Counter,deque\nimport time\n\n\ndef grid(value):\n    if not isinstance(value,(list,tuple)) or not 1<=len(value)<=30:raise ValueError(\'Invalid grid height\')\n    rows=tuple(tuple(row) for row in value)\n    if not rows[0] or len(rows[0])>30 or any(len(row)!=len(rows[0]) for row in rows):raise ValueError(\'Invalid grid width\')\n    if any(type(c) is not int or not 0<=c<=9 for row in rows for c in row):raise ValueError(\'Invalid grid colors\')\n    return rows\n\n\ndef modal(g):return Counter(c for row in g for c in row).most_common(1)[0][0]\n\n\ndef rotate(g):return tuple(tuple(row) for row in zip(*g[::-1]))\n\n\ndef orient(g,index):\n    if index>=4:g=tuple(tuple(row) for row in zip(*g));index-=4\n    for _ in range(index):g=rotate(g)\n    return g\n\n\ndef crop(g,color=None,background=None):\n    bg=modal(g) if background is None else background\n    points=[(r,c) for r,row in enumerate(g) for c,x in enumerate(row) if (x==color if color is not None else x!=bg)]\n    if not points:return None\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef object_crop(g,selection,monochrome,background):\n    bg=modal(g) if background is None else background\n    seen=set();objects=[];h=len(g);w=len(g[0])\n    for r in range(h):\n        for c in range(w):\n            if (r,c) in seen or g[r][c]==bg:continue\n            pending=deque([(r,c)]);seen.add((r,c));points=[]\n            while pending:\n                y,x=pending.popleft();points.append((y,x))\n                for yy,xx in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):\n                    if 0<=yy<h and 0<=xx<w and (yy,xx) not in seen and g[yy][xx]!=bg and (not monochrome or g[yy][xx]==g[r][c]):\n                        seen.add((yy,xx));pending.append((yy,xx))\n            objects.append(points)\n    if not objects:return None\n    objects.sort(key=lambda p:(len(p),min(p)))\n    points=objects[-1] if selection==\'largest\' else objects[0]\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef resize(g,scale,repeat=False):\n    if len(g)*scale>30 or len(g[0])*scale>30:return None\n    if repeat:return tuple(row*scale for _ in range(scale) for row in g)\n    return tuple(tuple(c for x in row for c in [x]*scale) for row in g for _ in range(scale))\n\n\ndef reduce_grid(g,scale):\n    if len(g)%scale or len(g[0])%scale:return None\n    result=[]\n    for r in range(0,len(g),scale):\n        row=[]\n        for c in range(0,len(g[0]),scale):\n            values={g[y][x] for y in range(r,r+scale) for x in range(c,c+scale)}\n            if len(values)!=1:return None\n            row.append(next(iter(values)))\n        result.append(tuple(row))\n    return tuple(result)\n\n\ndef compress(g):\n    rows=[]\n    for row in g:\n        if not rows or row!=rows[-1]:rows.append(row)\n    columns=[]\n    for col in zip(*rows):\n        if not columns or col!=columns[-1]:columns.append(col)\n    return tuple(tuple(row) for row in zip(*columns))\n\n\ndef halves(g,axis,mode,background):\n    if axis:g=orient(g,4)\n    n=len(g);middle=n//2\n    if n<2:return None\n    first=g[:middle];second=g[middle+n%2:]\n    bg=modal(g) if background is None else background\n    if mode==\'first\':out=first\n    elif mode==\'second\':out=second\n    else:\n        operations={\'or\':lambda a,b:a or b,\'and\':lambda a,b:a and b,\'xor\':lambda a,b:a!=b}\n        op=operations[mode]\n        out=tuple(tuple(int(op(a!=bg,b!=bg)) for a,b in zip(ra,rb)) for ra,rb in zip(first,second))\n    return orient(out,4) if axis else out\n\n\ndef operations(colors):\n    yield \'identity\',0,lambda g:g\n    yield \'compress_repeated_rows_columns\',1,compress\n    for bg in (None,0):\n        yield f\'foreground_bbox_{bg}\',1,lambda g,bg=bg:crop(g,background=bg)\n        for selection in (\'largest\',\'smallest\'):\n            for mono in (False,True):\n                yield f\'{selection}_object_{mono}_{bg}\',2,lambda g,s=selection,m=mono,b=bg:object_crop(g,s,m,b)\n        for axis in (0,1):\n            for mode in (\'first\',\'second\',\'or\',\'and\',\'xor\'):\n                yield f\'halves_{axis}_{mode}_{bg}\',2,lambda g,a=axis,m=mode,b=bg:halves(g,a,m,b)\n    for color in colors:yield f\'color_bbox_{color}\',2,lambda g,c=color:crop(g,color=c)\n    for scale in (2,3,4):\n        yield f\'upscale_{scale}\',1,lambda g,s=scale:resize(g,s)\n        yield f\'tile_{scale}\',2,lambda g,s=scale:resize(g,s,True)\n        yield f\'uniform_downscale_{scale}\',1,lambda g,s=scale:reduce_grid(g,s)\n\n\ndef fit_mapping(inputs,outputs):\n    mapping={}\n    for before,after in zip(inputs,outputs):\n        if before is None or len(before)!=len(after) or len(before[0])!=len(after[0]):return None\n        for ra,rb in zip(before,after):\n            for a,b in zip(ra,rb):\n                if a in mapping and mapping[a]!=b:return None\n                mapping[a]=b\n    return mapping\n\n\ndef solve(task,seconds=1.0):\n    started=time.monotonic();examples=task[\'train\'];tests=[grid(t[\'input\']) for t in task[\'test\']]\n    if not examples:return [dict(attempt_1=[list(row) for row in g],attempt_2=[list(row) for row in g]) for g in tests],dict(programs=0,reason=\'no demonstrations\')\n    inputs=[grid(t[\'input\']) for t in examples];outputs=[grid(t[\'output\']) for t in examples]\n    colors=sorted({c for g in inputs for row in g for c in row})\n    mappings={};prediction_sets=set();candidates=[];tried=0;timed_out=False\n    for name,cost,operation in operations(colors):\n        if time.monotonic()-started>=seconds:timed_out=True;break\n        transformed=[operation(g) for g in inputs]\n        if any(g is None for g in transformed):continue\n        for orientation in range(8):\n            views=tuple(orient(g,orientation) for g in transformed)\n            tried+=1\n            if views not in mappings:mappings[views]=fit_mapping(views,outputs)\n            mapping=mappings[views]\n            if mapping is None:continue\n            predictions=[]\n            for g in tests:\n                result=operation(g)\n                if result is None:break\n                result=orient(result,orientation)\n                result=tuple(tuple(mapping.get(c,c) for c in row) for row in result)\n                predictions.append(result)\n            if len(predictions)!=len(tests):continue\n            signature=tuple(predictions)\n            if signature in prediction_sets:continue\n            prediction_sets.add(signature)\n            recolors=sum(a!=b for a,b in mapping.items())\n            score=cost+0.1*(orientation!=0)+0.05*recolors\n            candidates.append((score,name,orientation,tuple(predictions)))\n    candidates.sort(key=lambda c:(c[0],c[1],c[2]))\n    attempts=[]\n    for i,g in enumerate(tests):\n        unique=[]\n        for candidate in candidates:\n            prediction=candidate[3][i]\n            if prediction not in unique:unique.append(prediction)\n            if len(unique)==2:break\n        if not unique:unique=[g]\n        if len(unique)==1:unique.append(g if g!=unique[0] else tuple(tuple(0 for _ in row) for row in g))\n        attempts.append(dict(attempt_1=[list(row) for row in unique[0]],attempt_2=[list(row) for row in unique[1]]))\n    return attempts,dict(programs=tried,consistent_programs=len(candidates),timed_out=timed_out,\n        elapsed_seconds=time.monotonic()-started,selected=[dict(name=c[1],orientation=c[2],complexity=c[0]) for c in candidates[:2]])\n', 'arc2/evaluation.py': '"""Official two-attempt exact-match metric, averaged over test outputs."""\nfrom .symbolic import grid\n\n\ndef validate_submission(challenges,submission):\n    if set(challenges)!=set(submission):raise ValueError(\'Submission IDs must exactly match challenge IDs\')\n    for key,task in challenges.items():\n        guesses=submission[key]\n        if not isinstance(guesses,list) or len(guesses)!=len(task[\'test\']):raise ValueError(\'Test-output count mismatch: \'+key)\n        for attempt in guesses:\n            if not isinstance(attempt,dict) or set(attempt)!=set((\'attempt_1\',\'attempt_2\')):\n                raise ValueError(\'Exactly two named attempts required\')\n            grid(attempt[\'attempt_1\']);grid(attempt[\'attempt_2\'])\n\n\ndef score_submission(challenges,solutions,submission):\n    validate_submission(challenges,submission)\n    if set(challenges)!=set(solutions):raise ValueError(\'Solution IDs differ\')\n    results=[]\n    for key,task in challenges.items():\n        if len(solutions[key])!=len(task[\'test\']):raise ValueError(\'Solution-output count mismatch\')\n        for index,target in enumerate(solutions[key]):\n            target=grid(target);attempt=submission[key][index]\n            correct=any(grid(attempt[name])==target for name in (\'attempt_1\',\'attempt_2\'))\n            results.append(dict(task_id=key,test_index=index,correct=correct))\n    return dict(correct_outputs=sum(r[\'correct\'] for r in results),test_outputs=len(results),\n        score=100*sum(r[\'correct\'] for r in results)/len(results) if results else 0,per_output=results)\n', 'arc2/neural.py': '"""Inference-only adapter for the public NVARC-derived 4B grid checkpoint.\n\nUses the public checkpoint\'s digit/chat serialization. This does not reproduce\nNVARC\'s LoRA test-time training, DFS decoding or augmentation rescoring.\n"""\nimport time\n\nfrom .symbolic import grid,orient\n\n\ndef grid_text(value):return \'\\n\'.join(\'\'.join(map(str,row)) for row in grid(value))\n\n\ndef prompt(task,index,orientation=0,skip=0):\n    parts=[]\n    for pair in task[\'train\'][skip:]:\n        parts.extend((\'<|im_start|>user\\n\',grid_text(orient(grid(pair[\'input\']),orientation)),\n            \'<|im_end|><|im_start|>assistant\\n\',grid_text(orient(grid(pair[\'output\']),orientation)),\'<|im_end|>\'))\n    parts.extend((\'<|im_start|>user\\n\',grid_text(orient(grid(task[\'test\'][index][\'input\']),orientation)),\n        \'<|im_end|><|im_start|>assistant\\n\'))\n    return \'\'.join(parts)\n\n\ndef parse_output(text):\n    text=text.split(\'<|im_end|>\',1)[0].strip(\'\\n\')\n    if not text or any(c not in \'0123456789\\n\' for c in text):raise ValueError(\'Expected digit grid only\')\n    return grid([[int(c) for c in row] for row in text.split(\'\\n\')])\n\n\ndef allowed_tokens(suffix,digits,newline,eos):\n    if eos in suffix:return [eos]\n    rows=[];current=[]\n    for token in suffix:\n        if token==newline:rows.append(current);current=[]\n        elif token in digits:current.append(token)\n        else:raise ValueError(\'Invalid generated grid token\')\n    if not rows:\n        result=list(digits) if len(current)<30 else []\n        if current:result.extend((newline,eos))\n        return result\n    width=len(rows[0])\n    if not 1<=width<=30 or any(len(row)!=width for row in rows):raise ValueError(\'Nonrectangular generated prefix\')\n    if len(current)<width:return list(digits)\n    if len(current)>width:raise ValueError(\'Generated row too wide\')\n    return [eos]+([newline] if len(rows)<29 else [])\n\n\nclass GridModel:\n    def __init__(self,path):\n        import torch\n        from transformers import AutoModelForCausalLM,AutoTokenizer\n        if not torch.cuda.is_available():raise RuntimeError(\'Actual CUDA GPU required before loading neural model\')\n        self.torch=torch\n        self.tokenizer=AutoTokenizer.from_pretrained(str(path),local_files_only=True,trust_remote_code=False)\n        self.model=AutoModelForCausalLM.from_pretrained(str(path),local_files_only=True,trust_remote_code=False,\n            use_safetensors=True,torch_dtype=torch.float16,device_map={\'\':0},low_cpu_mem_usage=True,\n            attn_implementation=\'sdpa\')\n        self.model.eval()\n        self.digits=[self.one_token(str(c)) for c in range(10)]\n        self.newline=self.one_token(\'\\n\');self.eos=self.tokenizer.eos_token_id\n        if self.eos is None:raise ValueError(\'Tokenizer has no EOS\')\n        ids=self.digits+[self.newline,self.eos]\n        if len(set(ids))!=12 or max(ids)>=self.model.config.vocab_size:\n            raise ValueError(\'Checkpoint grid-token vocabulary mismatch\')\n\n    def one_token(self,text):\n        ids=self.tokenizer.encode(text,add_special_tokens=False)\n        if len(ids)!=1:raise ValueError(\'Checkpoint requires atomic grid tokens\')\n        return ids[0]\n\n    def generate(self,task,index,orientation=0,deadline=None):\n        if orientation not in (0,4):raise ValueError(\'Only identity/transpose inference supported\')\n        if deadline is not None and time.monotonic()>=deadline-5:return None,dict(reason=\'budget\')\n        started=time.monotonic();skip=0\n        while True:\n            ids=self.tokenizer.encode(prompt(task,index,orientation,skip),add_special_tokens=False)\n            if len(ids)+960<=8192:break\n            if skip>=len(task[\'train\']):return None,dict(reason=\'context budget\')\n            skip+=1\n        if max(ids)>=self.model.config.vocab_size:raise ValueError(\'Prompt token exceeds checkpoint vocabulary\')\n        tokens=self.torch.tensor([ids],device=self.model.device,dtype=self.torch.long)\n        options=dict(max_new_tokens=960,do_sample=False,use_cache=True,eos_token_id=self.eos,pad_token_id=self.eos,\n            prefix_allowed_tokens_fn=lambda batch,seq:allowed_tokens(seq[len(ids):].tolist(),self.digits,self.newline,self.eos))\n        if deadline is not None:options[\'max_time\']=max(1.0,deadline-time.monotonic())\n        try:\n            with self.torch.inference_mode():output=self.model.generate(input_ids=tokens,**options)\n        except self.torch.OutOfMemoryError:\n            self.torch.cuda.empty_cache()\n            return None,dict(reason=\'GPU memory exhausted\',prompt_tokens=len(ids),orientation=orientation)\n        suffix=output[0,len(ids):].tolist()\n        info=dict(prompt_tokens=len(ids),generated_tokens=len(suffix),orientation=orientation,\n            demonstrations_omitted=skip,elapsed_seconds=time.monotonic()-started,terminated_with_eos=bool(suffix and suffix[-1]==self.eos))\n        if not info[\'terminated_with_eos\']:return None,dict(**info,reason=\'unfinished output\')\n        try:result=parse_output(self.tokenizer.decode(suffix,skip_special_tokens=False))\n        except ValueError:return None,dict(**info,reason=\'invalid grid\')\n        if orientation==4:result=orient(result,4)\n        return [list(row) for row in result],info\n', 'arc2/adaptation.py': '"""Small, resettable LoRA test-time adaptation on supplied demonstrations only."""\nimport random\nimport time\n\nfrom .neural import grid_text\nfrom .symbolic import grid,orient\n\n\ndef augmented_pair(pair,rng):\n    orientation=rng.randrange(8);permutation=list(range(10));rng.shuffle(permutation)\n    def transform(value):\n        return [[permutation[c] for c in row] for row in orient(grid(value),orientation)]\n    return dict(input=transform(pair[\'input\']),output=transform(pair[\'output\']))\n\n\ndef training_tokens(pair,tokenizer):\n    prefix=\'<|im_start|>user\\n\'+grid_text(pair[\'input\'])+\'<|im_end|><|im_start|>assistant\\n\'\n    suffix=grid_text(pair[\'output\'])+\'<|im_end|>\'\n    start=tokenizer.encode(prefix,add_special_tokens=False)\n    full=tokenizer.encode(prefix+suffix,add_special_tokens=False)\n    if full[:len(start)]!=start:raise ValueError(\'Tokenizer prefix boundary changed\')\n    if len(full)>2048:raise ValueError(\'Training pair exceeds adaptation context\')\n    return full,[-100]*len(start)+full[len(start):]\n\n\nclass TaskAdapter:\n    def __init__(self,backend,rank=16):\n        from peft import LoraConfig,get_peft_model,get_peft_model_state_dict,set_peft_model_state_dict\n        self.backend=backend;self.torch=backend.torch\n        self.get_state=get_peft_model_state_dict;self.set_state=set_peft_model_state_dict\n        config=LoraConfig(r=rank,lora_alpha=32,lora_dropout=0,bias=\'none\',task_type=\'CAUSAL_LM\',\n            target_modules=[\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'])\n        backend.model=get_peft_model(backend.model,config)\n        trainable=[(n,p) for n,p in backend.model.named_parameters() if p.requires_grad]\n        if not trainable or any(\'lora_\' not in n for n,p in trainable):raise ValueError(\'Only LoRA parameters may train\')\n        self.parameters=[p for n,p in trainable]\n        self.trainable_parameters=sum(p.numel() for p in self.parameters)\n        if self.trainable_parameters>50000000:raise ValueError(\'Adaptation parameter budget exceeded\')\n        self.initial={k:v.detach().cpu().clone() for k,v in self.get_state(backend.model).items()}\n        backend.model.enable_input_require_grads()\n        self.reset()\n\n    def reset(self):\n        self.set_state(self.backend.model,self.initial)\n        self.backend.model.eval()\n\n    def adapt(self,task,steps=8,deadline=None):\n        self.reset();model=self.backend.model;torch=self.torch\n        started=time.monotonic();rng=random.Random(42);losses=[];completed=0\n        optimizer=torch.optim.AdamW(self.parameters,lr=5e-5,weight_decay=0,foreach=False)\n        scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024)\n        model.config.use_cache=False\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n        model.train()\n        reason=\'complete\'\n        try:\n            for step in range(steps):\n                if deadline is not None and time.monotonic()>=deadline-3:reason=\'budget\';break\n                pair=augmented_pair(task[\'train\'][step%len(task[\'train\'])],rng)\n                ids,labels=training_tokens(pair,self.backend.tokenizer)\n                x=torch.tensor([ids],device=model.device,dtype=torch.long)\n                y=torch.tensor([labels],device=model.device,dtype=torch.long)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(\'cuda\',dtype=torch.float16):loss=model(input_ids=x,labels=y).loss\n                if not bool(torch.isfinite(loss)):reason=\'nonfinite loss\';break\n                scaler.scale(loss).backward();scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(self.parameters,1.0)\n                old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n                if scaler.get_scale()>=old_scale:completed+=1\n                losses.append(float(loss.detach().cpu()))\n        except torch.OutOfMemoryError:\n            reason=\'GPU memory exhausted\'\n        finally:\n            optimizer.zero_grad(set_to_none=True);del optimizer\n            model.gradient_checkpointing_disable();model.config.use_cache=True;model.eval()\n            torch.cuda.empty_cache()\n        return dict(steps_requested=steps,steps_completed=completed,losses=losses,stop_reason=reason,\n            elapsed_seconds=time.monotonic()-started,trainable_parameters=self.trainable_parameters,\n            source=\'Supplied demonstration input/output pairs only; no test targets\')\n\n    def change_from_initial(self):\n        current=self.get_state(self.backend.model)\n        return max(float((v.detach().cpu()-self.initial[k]).abs().max()) for k,v in current.items())\n'}
for name,source in sources.items():
    path=Path('/kaggle/working')/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source)
sys.path.insert(0,'/kaggle/working')
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True,stdout=subprocess.DEVNULL)
from arc2.symbolic import solve
from arc2.neural import GridModel
from arc2.adaptation import TaskAdapter
from arc2.evaluation import score_submission
roots=[Path('/kaggle/input/datasets/kunalsinghh03/arc2-public-training-development'),Path('/kaggle/input/arc2-public-training-development')]
root=next(p for p in roots if p.is_dir())
models=[Path('/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'),Path('/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1')]
backend=GridModel(next(p for p in models if (p/'config.json').is_file()));adapter=TaskAdapter(backend)
all_challenges=json.loads((root/'arc-agi_training_challenges.json').read_text())
challenges={k:all_challenges[k] for k in sorted(all_challenges)[:50]}
before={};after={};events=[]
def predict(task,fallback):
    result=[]
    for i,test in enumerate(task['test']):
        choices=[]
        for orientation in (0,4):
            prediction,info=backend.generate(task,i,orientation,deadline=min(END,time.monotonic()+120))
            if prediction is not None and prediction not in choices:choices.append(prediction)
        for prediction in fallback[i].values():
            if prediction not in choices:choices.append(prediction)
        result.append(dict(attempt_1=choices[0],attempt_2=choices[1] if len(choices)>1 else choices[0]))
    return result
for number,(key,task) in enumerate(challenges.items()):
    floor,_=solve(task,seconds=0.25)
    adapter.reset();before[key]=predict(task,floor)
    adaptation=adapter.adapt(task,steps=8,deadline=min(END,time.monotonic()+60))
    after[key]=predict(task,floor)
    events.append(dict(task_id=key,adaptation=adaptation))
    if (number+1)%5==0:
        Path('/kaggle/working/before-predictions.json').write_text(json.dumps(before))
        Path('/kaggle/working/after-predictions.json').write_text(json.dumps(after))
        print('PAIRED_ADAPTATION_PROGRESS',number+1,len(challenges),time.monotonic()-START,flush=True)
# Predictions are frozen before opening the public TRAINING solution labels.
Path('/kaggle/working/before-predictions.json').write_text(json.dumps(before))
Path('/kaggle/working/after-predictions.json').write_text(json.dumps(after))
all_targets=json.loads((root/'arc-agi_training_solutions.json').read_text());targets={k:all_targets[k] for k in challenges}
b=score_submission(challenges,targets,before);a=score_submission(challenges,targets,after)
improved=sum(not x['correct'] and y['correct'] for x,y in zip(b['per_output'],a['per_output']))
worsened=sum(x['correct'] and not y['correct'] for x,y in zip(b['per_output'],a['per_output']))
b.pop('per_output');a.pop('per_output')
report=dict(tasks=len(challenges),before=b,after=a,improved_outputs=improved,worsened_outputs=worsened,
    elapsed_seconds=time.monotonic()-START,events=events,
    boundary='Matched single deterministic pilot on 50 public training tasks. Pretraining may include these tasks; no unseen generalization, official improvement or statistical significance established.')
Path('/kaggle/working/adaptation-comparison.json').write_text(json.dumps(report,indent=2))
print('ARC2_ADAPTATION_COMPARISON_COMPLETE',json.dumps({k:v for k,v in report.items() if k!='events'}),flush=True)
